# AERONET Deep Dive: Boundary Layer Decoupling & the Addis Anomaly

**Follow-up to `aeronet_addis_diagnostics.ipynb`** — this notebook refines the key findings and connects them to the 24-hour filter data (HIPS, FTIR, Iron).

## Headline Finding
The Addis anomaly is largely explained by **strong boundary layer dynamics at 2370m elevation** causing systematic surface-column decoupling. Surface BC is highly variable and morning-biased due to BL trapping, while the column integrates over a deeper mixed layer.

## Structure
1. **Diurnal decoupling** — BC drops 6am→midday while AOD stays flat or rises (the publishable figure)
2. **BC/AOD ratio as BL depth proxy** — 3-4x morning-to-afternoon change
3. **Filter data × AERONET** — 24h-average HIPS/FTIR/BC matched with daily AERONET
4. **Dust interference null result** — HIPS/FTIR ratio vs coarse AOD (negative correlation)
5. **Kiremt coarse mode anomaly** — why dust peaks in the rainy season
6. **AAE data quality** — investigating negative surface AAE values
7. **Regime synthesis** — connecting FMF×BC regimes to MAC suppression

---

## Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

RESEARCH_DIR = '/Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem'
scripts_path = os.path.join(RESEARCH_DIR, 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, FILTER_DATA_PATH, PROCESSED_SITES_DIR, MAC_VALUE
from data_matching import (
    load_aethalometer_data, load_filter_data,
    match_all_parameters, pivot_filter_by_id
)

# Addis season definitions
SEASONS = {'Dry': [10, 11, 12, 1, 2], 'Belg': [3, 4, 5], 'Kiremt': [6, 7, 8, 9]}
SEASON_COLORS = {'Dry': '#E67E22', 'Belg': '#27AE60', 'Kiremt': '#3498DB'}
SEASON_ORDER = ['Dry', 'Belg', 'Kiremt']

def assign_season(month):
    for name, months in SEASONS.items():
        if month in months:
            return name
    return 'Unknown'

AERONET_MISSING = -999.
SITE_CODE = 'ETAD'

print('Setup complete')

## Data Loading

In [ ]:
# === Paths ===
BC_MINUTE_PATH = (
    '/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/'
    'My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/'
    'Aethelometry Data/JacrosMA350 60s Data20250804082112/'
    'df_jacros_cleaned_API_and_OG_manual_BC_all_wl.pkl'
)

AERONET_BASE = (
    '/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/'
    'My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/'
    'AERONET/Jacros'
)
AERONET_AOD_ALLPTS = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET all',
    '20220101_20251231_AAU_Jackros_ET.lev20')
AERONET_SDA_ALLPTS = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET all',
    '20220101_20251231_AAU_Jackros_ET.ONEILL_lev20')
AERONET_AOD_DAILY = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET Daily',
    '20220101_20251231_AAU_Jackros_ET.lev20')
AERONET_SDA_DAILY = os.path.join(
    AERONET_BASE, '20220101_20251231_AAU_Jackros_ET Daily',
    '20220101_20251231_AAU_Jackros_ET.ONEILL_lev20')

print('Paths configured')

In [ ]:
# === Load minute-level BC ===
bc_raw = pd.read_pickle(BC_MINUTE_PATH)
bc_raw['datetime_local'] = pd.to_datetime(bc_raw['datetime_local'])
bc_raw.set_index('datetime_local', inplace=True)
bc_raw.sort_index(inplace=True)

for col in ['UV BCc', 'IR BCc', 'UV BC1', 'IR BC1']:
    if col in bc_raw.columns:
        bc_raw[col] = bc_raw[col] / 1000  # ng/m³ → µg/m³

for col in ['UV BCc', 'IR BCc']:
    if col in bc_raw.columns:
        bc_raw.loc[bc_raw[col] < 0, col] = np.nan
        mu, sigma = bc_raw[col].mean(), bc_raw[col].std()
        bc_raw.loc[bc_raw[col] > mu + 3 * sigma, col] = np.nan

print(f'BC minute data: {len(bc_raw):,} records')
print(f'  Range: {bc_raw.index.min()} → {bc_raw.index.max()}')

In [ ]:
# === Load AERONET all-points (sub-daily) ===
def load_aeronet_allpoints(filepath, date_col='Date(dd:mm:yyyy)', time_col='Time(hh:mm:ss)'):
    df = pd.read_csv(filepath, skiprows=6)
    df['datetime_utc'] = pd.to_datetime(
        df[date_col] + ' ' + df[time_col], format='%d:%m:%Y %H:%M:%S')
    df.set_index('datetime_utc', inplace=True)
    df.sort_index(inplace=True)
    df.replace(AERONET_MISSING, np.nan, inplace=True)
    return df

aod_all = load_aeronet_allpoints(AERONET_AOD_ALLPTS)
sda_all = load_aeronet_allpoints(AERONET_SDA_ALLPTS,
                                  date_col='Date_(dd:mm:yyyy)', time_col='Time_(hh:mm:ss)')
print(f'AERONET AOD all-points: {len(aod_all):,}')
print(f'AERONET SDA all-points: {len(sda_all):,}')

In [ ]:
# === Load daily AERONET ===
def load_aeronet_daily(filepath, date_col='Date(dd:mm:yyyy)'):
    df = pd.read_csv(filepath, skiprows=6)
    df['Date'] = pd.to_datetime(df[date_col], format='%d:%m:%Y')
    df.set_index('Date', inplace=True)
    df.sort_index(inplace=True)
    df.replace(AERONET_MISSING, np.nan, inplace=True)
    return df

aod_daily = load_aeronet_daily(AERONET_AOD_DAILY)
sda_daily = load_aeronet_daily(AERONET_SDA_DAILY, date_col='Date_(dd:mm:yyyy)')
sda_daily['month'] = sda_daily.index.month
sda_daily['season'] = sda_daily['month'].map(assign_season)
aod_daily['month'] = aod_daily.index.month
aod_daily['season'] = aod_daily['month'].map(assign_season)

print(f'AERONET daily AOD: {len(aod_daily)} days')
print(f'AERONET daily SDA: {len(sda_daily)} days')

In [ ]:
# === Load 24h filter data using existing match_all_parameters ===
aeth_data = load_aethalometer_data()
filter_data = load_filter_data()

df_matched_filters = match_all_parameters('Addis_Ababa', SITE_CODE,
                                           aeth_data['Addis_Ababa'], filter_data)

if df_matched_filters is not None:
    df_matched_filters['date'] = pd.to_datetime(df_matched_filters['date'])
    df_matched_filters['month'] = df_matched_filters['date'].dt.month
    df_matched_filters['season'] = df_matched_filters['month'].map(assign_season)
    print(f'\nMatched filter samples: {len(df_matched_filters)}')
    print(f'  Date range: {df_matched_filters["date"].min().date()} → {df_matched_filters["date"].max().date()}')
    print(f'  Columns: {df_matched_filters.columns.tolist()}')
    for col in ['ir_bcc', 'hips_fabs', 'ftir_ec', 'iron']:
        if col in df_matched_filters.columns:
            n = df_matched_filters[col].notna().sum()
            print(f'  {col}: {n} valid')
else:
    print('WARNING: No matched filter data')

In [ ]:
# === Sub-daily matching: AERONET obs × ±15min BC ===
# (reuse matched data from diagnostics notebook, or recompute)

# Convert BC to UTC
bc_utc = bc_raw[['IR BCc', 'UV BCc']].copy()
if bc_utc.index.tz is not None:
    bc_utc.index = bc_utc.index.tz_convert('UTC').tz_localize(None)
else:
    bc_utc.index = bc_utc.index - pd.Timedelta(hours=3)

# Merge AOD + SDA on shared timestamps
aod_cols = ['AOD_500nm', 'AOD_440nm', 'AOD_870nm', 'Precipitable_Water(cm)',
            '440-870_Angstrom_Exponent', 'Solar_Zenith_Angle(Degrees)']
sda_cols = ['Total_AOD_500nm[tau_a]', 'Fine_Mode_AOD_500nm[tau_f]',
            'Coarse_Mode_AOD_500nm[tau_c]', 'FineModeFraction_500nm[eta]',
            'Angstrom_Exponent(AE)-Total_500nm[alpha]']

aeronet_merged = aod_all[aod_cols].join(sda_all[sda_cols], how='outer')

MATCH_WINDOW = pd.Timedelta(minutes=15)
matched_rows = []

for t_aero, row in aeronet_merged.iterrows():
    mask = (bc_utc.index >= t_aero - MATCH_WINDOW) & (bc_utc.index <= t_aero + MATCH_WINDOW)
    bc_window = bc_utc.loc[mask]
    if len(bc_window) < 5:
        continue
    ir_bc = bc_window['IR BCc'].mean()
    uv_bc = bc_window['UV BCc'].mean()
    if pd.isna(ir_bc):
        continue
    rec = {
        'datetime_utc': t_aero,
        'datetime_local': t_aero + pd.Timedelta(hours=3),
        'IR_BCc': ir_bc, 'UV_BCc': uv_bc,
        'n_bc_minutes': bc_window['IR BCc'].notna().sum(),
    }
    for col in aod_cols + sda_cols:
        rec[col] = row.get(col, np.nan)
    matched_rows.append(rec)

matched = pd.DataFrame(matched_rows)
matched['hour_local'] = matched['datetime_local'].dt.hour
matched['month'] = matched['datetime_local'].dt.month
matched['season'] = matched['month'].map(assign_season)
matched['date'] = matched['datetime_local'].dt.date

print(f'Sub-daily matched: {len(matched):,} observations, {matched["date"].nunique()} days')
print(matched['season'].value_counts())

---

## 1. Diurnal Decoupling — The Headline Finding

In all three seasons, BC drops sharply from 6am → midday while AOD stays flat or rises. This is textbook boundary layer behavior:
- **Morning**: shallow BL traps emissions near surface → high BC, but column AOD is low because the aerosol is concentrated in a thin layer
- **Midday**: BL grows and mixes → BC dilutes through a deeper column, AOD either holds or rises as you see more atmosphere

This is strong physical evidence that morning surface BC spikes are a **trapping artifact**, not a true emissions spike.

In [ ]:
# Publication-quality diurnal decoupling figure
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=False)

for idx, season in enumerate(SEASON_ORDER):
    ax = axes[idx]
    sdata = matched[matched['season'] == season]
    if len(sdata) < 10:
        ax.set_title(f'{season} (insufficient data)')
        continue

    hourly_bc = sdata.groupby('hour_local')['IR_BCc'].agg(['mean', 'std', 'count'])
    hourly_aod = sdata.groupby('hour_local')['AOD_500nm'].agg(['mean', 'std', 'count'])

    ax2 = ax.twinx()

    # BC (left axis)
    bc_se = hourly_bc['std'] / np.sqrt(hourly_bc['count'])
    ax.fill_between(hourly_bc.index, hourly_bc['mean'] - bc_se,
                    hourly_bc['mean'] + bc_se, alpha=0.2, color='#E74C3C')
    ax.plot(hourly_bc.index, hourly_bc['mean'], 'o-', color='#E74C3C',
            linewidth=2.5, markersize=7, label='Surface BC', zorder=5)

    # AOD (right axis)
    valid_aod = hourly_aod.dropna(subset=['mean'])
    aod_se = valid_aod['std'] / np.sqrt(valid_aod['count'])
    ax2.fill_between(valid_aod.index, valid_aod['mean'] - aod_se,
                     valid_aod['mean'] + aod_se, alpha=0.2, color='#3498DB')
    ax2.plot(valid_aod.index, valid_aod['mean'], 's-', color='#3498DB',
             linewidth=2.5, markersize=7, label='AOD 500nm', zorder=5)

    # Add observation counts
    for h in hourly_bc.index:
        ax.text(h, ax.get_ylim()[0], f'n={int(hourly_bc.loc[h, "count"])}',
                fontsize=6, ha='center', va='bottom', alpha=0.5)

    ax.set_xlabel('Hour (local time)', fontsize=12)
    if idx == 0:
        ax.set_ylabel('Surface BC (µg/m³)', fontsize=12, color='#E74C3C')
    if idx == 2:
        ax2.set_ylabel('AOD 500nm', fontsize=12, color='#3498DB')
    ax.set_title(f'{season} Season (n={len(sdata):,})', fontsize=13, fontweight='bold')

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='y', colors='#E74C3C')
    ax2.tick_params(axis='y', colors='#3498DB')

plt.suptitle('Diurnal Decoupling: Surface BC vs Columnar AOD at Addis Ababa (2370m)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Print the decoupling strength
print('\nBC change from early morning (6-8) to midday (11-13):')
for season in SEASON_ORDER:
    sdata = matched[matched['season'] == season]
    morning = sdata[sdata['hour_local'].between(6, 8)]['IR_BCc'].mean()
    midday = sdata[sdata['hour_local'].between(11, 13)]['IR_BCc'].mean()
    aod_morning = sdata[sdata['hour_local'].between(6, 8)]['AOD_500nm'].mean()
    aod_midday = sdata[sdata['hour_local'].between(11, 13)]['AOD_500nm'].mean()
    if not np.isnan(morning) and morning > 0:
        print(f'  {season}: BC {morning:.2f} → {midday:.2f} ({(midday-morning)/morning*100:+.0f}%), '
              f'AOD {aod_morning:.3f} → {aod_midday:.3f} ({(aod_midday-aod_morning)/aod_morning*100:+.0f}%)')

---

## 2. BC/AOD Ratio — Boundary Layer Depth Proxy

The BC/AOD ratio quantifies how much of the columnar aerosol is at the surface. A high ratio means aerosol is concentrated near the ground (shallow BL). The ~3-4x morning-to-afternoon swing is a direct measurement of BL growth.

In [ ]:
# BC/AOD ratio analysis
valid_match = matched.dropna(subset=['IR_BCc', 'AOD_500nm']).copy()
valid_match = valid_match[valid_match['AOD_500nm'] > 0.01]
valid_match['BC_AOD_ratio'] = valid_match['IR_BCc'] / valid_match['AOD_500nm']

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: Diurnal BC/AOD ratio by season
ax = axes[0]
for season in SEASON_ORDER:
    sdata = valid_match[valid_match['season'] == season]
    hourly = sdata.groupby('hour_local')['BC_AOD_ratio'].agg(['mean', 'std', 'count'])
    se = hourly['std'] / np.sqrt(hourly['count'])
    ax.fill_between(hourly.index, hourly['mean'] - se, hourly['mean'] + se,
                    alpha=0.15, color=SEASON_COLORS[season])
    ax.plot(hourly.index, hourly['mean'], 'o-', color=SEASON_COLORS[season],
            linewidth=2.5, markersize=7, label=season)

ax.set_xlabel('Hour (local time)', fontsize=12)
ax.set_ylabel('BC / AOD$_{500}$ ratio', fontsize=12)
ax.set_title('BC-to-AOD Ratio Diurnal Cycle\n(higher = more surface-confined)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Panel 2: Morning (6-9) vs Afternoon (13-16) BC-AOD slopes
ax = axes[1]
valid_match['period'] = 'midday'
valid_match.loc[valid_match['hour_local'].between(6, 9), 'period'] = 'morning (6-9h)'
valid_match.loc[valid_match['hour_local'].between(13, 16), 'period'] = 'afternoon (13-16h)'

for period, marker, color in [('morning (6-9h)', 'o', '#E74C3C'),
                                ('afternoon (13-16h)', '^', '#3498DB')]:
    pdata = valid_match[valid_match['period'] == period]
    ax.scatter(pdata['IR_BCc'], pdata['AOD_500nm'], marker=marker,
              alpha=0.3, s=20, color=color, label=f'{period} (n={len(pdata)})')
    clean = pdata[['IR_BCc', 'AOD_500nm']].dropna()
    if len(clean) > 10:
        slope, intercept, r, p, _ = stats.linregress(clean['IR_BCc'], clean['AOD_500nm'])
        x_fit = np.linspace(clean['IR_BCc'].min(), clean['IR_BCc'].max(), 50)
        ax.plot(x_fit, slope * x_fit + intercept, '-', color=color, linewidth=2.5)
        ax.text(0.05, 0.95 if period.startswith('m') else 0.85,
                f'{period}:\nslope={slope:.4f}, r={r:.3f}',
                transform=ax.transAxes, fontsize=8, va='top',
                color=color, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('BC (µg/m³)', fontsize=12)
ax.set_ylabel('AOD 500nm', fontsize=12)
ax.set_title('Morning vs Afternoon BC-AOD Relationship\n(steeper slope = better coupling)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

# Panel 3: BC/AOD ratio distribution by season
ax = axes[2]
plot_data = [valid_match[valid_match['season'] == s]['BC_AOD_ratio'].dropna() for s in SEASON_ORDER]
bp = ax.boxplot(plot_data, labels=SEASON_ORDER, patch_artist=True, showfliers=False,
               widths=0.6)
for patch, season in zip(bp['boxes'], SEASON_ORDER):
    patch.set_facecolor(SEASON_COLORS[season])
    patch.set_alpha(0.7)
for i, season in enumerate(SEASON_ORDER):
    n = len(valid_match[valid_match['season'] == season])
    ax.text(i + 1, ax.get_ylim()[1] * 0.95, f'n={n}', ha='center', fontsize=9)
ax.set_ylabel('BC / AOD$_{500}$ ratio', fontsize=12)
ax.set_title('BC/AOD Ratio by Season\n(Kiremt lower = better mixed)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('BC/AOD Ratio as Boundary Layer Depth Proxy', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nBC/AOD ratio by season and time of day:')
for season in SEASON_ORDER:
    sdata = valid_match[valid_match['season'] == season]
    am = sdata[sdata['period'] == 'morning (6-9h)']['BC_AOD_ratio']
    pm = sdata[sdata['period'] == 'afternoon (13-16h)']['BC_AOD_ratio']
    print(f'  {season}: morning median={am.median():.1f}, afternoon median={pm.median():.1f}, '
          f'ratio={am.median()/pm.median():.1f}x')

---

## 3. Filter Data × AERONET — Connecting 24h Averages to the Column

The 24-hour filter samples (HIPS Fabs, FTIR EC, XRF Iron) integrate over a full day. We match these with daily AERONET retrievals to connect surface filter measurements directly to columnar properties.

In [ ]:
# Merge 24h filter data with daily AERONET
coarse_col = 'Coarse_Mode_AOD_500nm[tau_c]'
fine_col = 'Fine_Mode_AOD_500nm[tau_f]'
fmf_col = 'FineModeFraction_500nm[eta]'

# Prepare daily AERONET for merge
aeronet_daily_merge = aod_daily[['AOD_500nm', 'Precipitable_Water(cm)',
                                  '440-870_Angstrom_Exponent']].copy()
sda_daily_merge = sda_daily[[coarse_col, fine_col, fmf_col]].copy()
aeronet_daily_combined = aeronet_daily_merge.join(sda_daily_merge, how='outer')

# Merge filter data with daily AERONET by date
if df_matched_filters is not None:
    filt_aero = df_matched_filters.copy()
    filt_aero['date_key'] = filt_aero['date'].dt.normalize()
    aeronet_daily_combined.index = pd.to_datetime(aeronet_daily_combined.index)

    filt_aero = filt_aero.merge(
        aeronet_daily_combined, left_on='date_key', right_index=True, how='left')

    n_with_aeronet = filt_aero['AOD_500nm'].notna().sum()
    n_with_coarse = filt_aero[coarse_col].notna().sum()
    print(f'Filter samples with AERONET AOD: {n_with_aeronet} / {len(filt_aero)}')
    print(f'Filter samples with coarse AOD: {n_with_coarse} / {len(filt_aero)}')

    # Compute HIPS EC equivalent and discrepancy
    if 'hips_fabs' in filt_aero.columns and 'ftir_ec' in filt_aero.columns:
        # hips_fabs is already in µg/m³ (divided by MAC in match_all_parameters)
        filt_aero['hips_ec'] = filt_aero['hips_fabs']
        valid_both = filt_aero.dropna(subset=['hips_ec', 'ftir_ec'])
        valid_both = valid_both[valid_both['ftir_ec'] > 0]
        filt_aero.loc[valid_both.index, 'hips_ftir_ratio'] = (
            valid_both['hips_ec'] / valid_both['ftir_ec'])
        filt_aero.loc[valid_both.index, 'hips_ftir_diff'] = (
            valid_both['hips_ec'] - valid_both['ftir_ec'])
        print(f'Samples with both HIPS and FTIR: {len(valid_both)}')
        print(f'  HIPS/FTIR ratio: mean={valid_both["hips_ec"].mean() / valid_both["ftir_ec"].mean():.2f}')
else:
    filt_aero = None
    print('No filter data available')

In [ ]:
# Filter data overview: HIPS, FTIR, BC, Iron vs AERONET properties
if filt_aero is not None:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Row 1: Surface measurements vs AOD
    filter_vs_aod = [
        ('ir_bcc', 'Aeth BC (µg/m³)'),
        ('hips_fabs', 'HIPS Fabs/MAC (µg/m³)'),
        ('ftir_ec', 'FTIR EC (µg/m³)'),
    ]
    for i, (col, label) in enumerate(filter_vs_aod):
        ax = axes[0, i]
        valid = filt_aero.dropna(subset=[col, 'AOD_500nm'])
        if len(valid) > 3:
            for season in SEASON_ORDER:
                s = valid[valid['season'] == season]
                ax.scatter(s[col], s['AOD_500nm'], color=SEASON_COLORS[season],
                          alpha=0.6, s=40, label=season, edgecolors='k', linewidth=0.3)
            if len(valid) > 5:
                r, p = stats.pearsonr(valid[col], valid['AOD_500nm'])
                ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}\nn={len(valid)}',
                        transform=ax.transAxes, fontsize=9, va='top',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        ax.set_xlabel(label, fontsize=10)
        ax.set_ylabel('AOD 500nm' if i == 0 else '', fontsize=10)
        ax.set_title(f'{label.split("(")[0].strip()} vs AOD', fontsize=11, fontweight='bold')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    # Row 2: Surface measurements vs Fine/Coarse AOD
    filter_vs_sda = [
        ('ir_bcc', 'Aeth BC (µg/m³)', fine_col, 'Fine AOD'),
        ('hips_fabs', 'HIPS EC (µg/m³)', coarse_col, 'Coarse AOD'),
        ('iron', 'XRF Iron (µg/m³)', coarse_col, 'Coarse AOD'),
    ]
    for i, (xcol, xlabel, ycol, ylabel) in enumerate(filter_vs_sda):
        ax = axes[1, i]
        valid = filt_aero.dropna(subset=[xcol, ycol])
        if len(valid) > 3:
            for season in SEASON_ORDER:
                s = valid[valid['season'] == season]
                ax.scatter(s[xcol], s[ycol], color=SEASON_COLORS[season],
                          alpha=0.6, s=40, label=season, edgecolors='k', linewidth=0.3)
            if len(valid) > 5:
                r, p = stats.pearsonr(valid[xcol], valid[ycol])
                ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}\nn={len(valid)}',
                        transform=ax.transAxes, fontsize=9, va='top',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_ylabel(ylabel if i == 0 or ylabel != axes[1, i-1].get_ylabel() else '', fontsize=10)
        ax.set_title(f'{xlabel.split("(")[0].strip()} vs {ylabel}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    plt.suptitle('24h Filter Measurements vs Daily AERONET Columnar Properties',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

---

## 4. Dust Interference Null Result

**Important finding**: HIPS/FTIR ratio vs coarse AOD shows r ≈ −0.33 — statistically significant but **negative**. Higher dust loading corresponds to HIPS *under*-estimating relative to FTIR. If dust caused absorption interference in HIPS, we'd expect the ratio to go *up* with dust. This argues **against** dust being the main driver of the HIPS anomaly.

In [ ]:
if filt_aero is not None and 'hips_ftir_ratio' in filt_aero.columns:
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    # Panel 1: HIPS/FTIR ratio vs Coarse AOD (the null result)
    ax = axes[0]
    valid = filt_aero.dropna(subset=['hips_ftir_ratio', coarse_col])
    if len(valid) > 3:
        for season in SEASON_ORDER:
            s = valid[valid['season'] == season]
            ax.scatter(s[coarse_col], s['hips_ftir_ratio'], color=SEASON_COLORS[season],
                      alpha=0.6, s=50, label=season, edgecolors='k', linewidth=0.3)
        if len(valid) > 5:
            r, p = stats.pearsonr(valid[coarse_col], valid['hips_ftir_ratio'])
            slope, intercept, _, _, _ = stats.linregress(valid[coarse_col], valid['hips_ftir_ratio'])
            x_fit = np.linspace(valid[coarse_col].min(), valid[coarse_col].max(), 50)
            ax.plot(x_fit, slope * x_fit + intercept, 'k--', linewidth=2, alpha=0.7)
            ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}\nn={len(valid)}',
                    transform=ax.transAxes, fontsize=10, va='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.axhline(y=1, color='gray', linestyle=':', alpha=0.7, label='1:1')
    ax.set_xlabel('Coarse Mode AOD 500nm (dust proxy)', fontsize=11)
    ax.set_ylabel('HIPS EC / FTIR EC', fontsize=11)
    ax.set_title('HIPS-FTIR Discrepancy vs Dust\n(negative r = dust NOT the cause)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 2: HIPS/FTIR ratio vs FMF
    ax = axes[1]
    valid_fmf = filt_aero.dropna(subset=['hips_ftir_ratio', fmf_col])
    if len(valid_fmf) > 3:
        for season in SEASON_ORDER:
            s = valid_fmf[valid_fmf['season'] == season]
            ax.scatter(s[fmf_col], s['hips_ftir_ratio'], color=SEASON_COLORS[season],
                      alpha=0.6, s=50, label=season, edgecolors='k', linewidth=0.3)
        if len(valid_fmf) > 5:
            r, p = stats.pearsonr(valid_fmf[fmf_col], valid_fmf['hips_ftir_ratio'])
            ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}', transform=ax.transAxes,
                    fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.axhline(y=1, color='gray', linestyle=':', alpha=0.7)
    ax.set_xlabel('Fine Mode Fraction (500nm)', fontsize=11)
    ax.set_ylabel('HIPS EC / FTIR EC', fontsize=11)
    ax.set_title('HIPS-FTIR Discrepancy vs FMF', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 3: HIPS/FTIR ratio vs total AOD
    ax = axes[2]
    valid_aod = filt_aero.dropna(subset=['hips_ftir_ratio', 'AOD_500nm'])
    if len(valid_aod) > 3:
        for season in SEASON_ORDER:
            s = valid_aod[valid_aod['season'] == season]
            ax.scatter(s['AOD_500nm'], s['hips_ftir_ratio'], color=SEASON_COLORS[season],
                      alpha=0.6, s=50, label=season, edgecolors='k', linewidth=0.3)
        if len(valid_aod) > 5:
            r, p = stats.pearsonr(valid_aod['AOD_500nm'], valid_aod['hips_ftir_ratio'])
            ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}', transform=ax.transAxes,
                    fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.axhline(y=1, color='gray', linestyle=':', alpha=0.7)
    ax.set_xlabel('Total AOD 500nm', fontsize=11)
    ax.set_ylabel('HIPS EC / FTIR EC', fontsize=11)
    ax.set_title('HIPS-FTIR Discrepancy vs Total AOD', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 4: Iron (surface) vs Coarse AOD (column) — do dust proxies agree?
    ax = axes[3]
    if 'iron' in filt_aero.columns:
        valid_iron = filt_aero.dropna(subset=['iron', coarse_col])
        if len(valid_iron) > 3:
            for season in SEASON_ORDER:
                s = valid_iron[valid_iron['season'] == season]
                ax.scatter(s[coarse_col], s['iron'], color=SEASON_COLORS[season],
                          alpha=0.6, s=50, label=season, edgecolors='k', linewidth=0.3)
            if len(valid_iron) > 5:
                r, p = stats.pearsonr(valid_iron[coarse_col], valid_iron['iron'])
                ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}\nn={len(valid_iron)}',
                        transform=ax.transAxes, fontsize=10, va='top',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.set_xlabel('Coarse Mode AOD 500nm', fontsize=11)
    ax.set_ylabel('XRF Iron (µg/m³)', fontsize=11)
    ax.set_title('Columnar vs Surface Dust Proxy\n(do they agree?)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.suptitle('Dust Interference Investigation: HIPS-FTIR Discrepancy vs AERONET Dust Indicators',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Insufficient data for dust interference analysis')

---

## 5. Kiremt Coarse Mode Anomaly

**Surprise**: Coarse-mode AOD peaks during Kiremt (rainy season), not Dry. This is counterintuitive for dust. Possible explanations:
- Moisture-laden Indian Ocean air masses carrying coarse aerosol
- Rain-triggered construction/soil disturbance
- Saharan dust transport corridor shifting with ITCZ

This connects to the regime analysis: Kiremt is dominated by the "High BC + Low FMF" regime (surface BC trapped below a coarse-dominated column).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: Coarse vs Fine AOD time series
ax = axes[0]
for col, label, color in [(fine_col, 'Fine Mode', '#3498DB'),
                           (coarse_col, 'Coarse Mode', '#8B4513')]:
    valid = sda_daily[col].dropna()
    ax.plot(valid.index, valid.values, 'o', markersize=2, alpha=0.3, color=color)
    rolling = valid.rolling(30, min_periods=7).mean()
    ax.plot(rolling.index, rolling.values, '-', linewidth=2.5, color=color, label=label)

# Shade Kiremt periods
for year in range(2022, 2025):
    kiremt_start = pd.Timestamp(f'{year}-06-01')
    kiremt_end = pd.Timestamp(f'{year}-09-30')
    ax.axvspan(kiremt_start, kiremt_end, alpha=0.1, color='blue')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('AOD 500nm', fontsize=11)
ax.set_title('Fine & Coarse Mode AOD Time Series\n(blue shading = Kiremt rainy season)', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Panel 2: Monthly Coarse AOD
ax = axes[1]
valid_sda = sda_daily.dropna(subset=[coarse_col])
monthly_coarse = valid_sda.groupby(valid_sda.index.month)[coarse_col].agg(['mean', 'std', 'count'])
monthly_coarse.index = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
colors = []
for m in range(1, 13):
    s = assign_season(m)
    colors.append(SEASON_COLORS[s])

bars = ax.bar(range(12), monthly_coarse['mean'], yerr=monthly_coarse['std'] / np.sqrt(monthly_coarse['count']),
              color=colors, alpha=0.8, capsize=3, edgecolor='k', linewidth=0.5)
ax.set_xticks(range(12))
ax.set_xticklabels(monthly_coarse.index, rotation=45, fontsize=9)
ax.set_ylabel('Coarse Mode AOD 500nm', fontsize=11)
ax.set_title('Monthly Coarse AOD\n(peaks in Kiremt, not Dry)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Panel 3: FMF monthly
ax = axes[2]
monthly_fmf = valid_sda.groupby(valid_sda.index.month)[fmf_col].agg(['mean', 'std', 'count'])
monthly_fmf.index = monthly_coarse.index
bars = ax.bar(range(12), monthly_fmf['mean'], yerr=monthly_fmf['std'] / np.sqrt(monthly_fmf['count']),
              color=colors, alpha=0.8, capsize=3, edgecolor='k', linewidth=0.5)
ax.set_xticks(range(12))
ax.set_xticklabels(monthly_fmf.index, rotation=45, fontsize=9)
ax.set_ylabel('Fine Mode Fraction (500nm)', fontsize=11)
ax.set_title('Monthly Fine Mode Fraction\n(lowest in Kiremt = coarse-dominated)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Kiremt Coarse Mode Anomaly: Dust Peaks in the Rainy Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCoarse Mode AOD by season:')
for season in SEASON_ORDER:
    s = valid_sda[valid_sda['season'] == season][coarse_col]
    print(f'  {season}: mean={s.mean():.4f}, median={s.median():.4f}, n={len(s)}')

In [ ]:
# Connect to filter data: is the HIPS anomaly worse when coarse mode is high?
if filt_aero is not None and 'hips_ftir_ratio' in filt_aero.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: HIPS/FTIR ratio by season
    ax = axes[0]
    valid = filt_aero.dropna(subset=['hips_ftir_ratio'])
    plot_data = [valid[valid['season'] == s]['hips_ftir_ratio'].dropna() for s in SEASON_ORDER]
    bp = ax.boxplot(plot_data, labels=SEASON_ORDER, patch_artist=True, showfliers=True,
                   flierprops=dict(marker='o', markersize=3, alpha=0.3))
    for patch, season in zip(bp['boxes'], SEASON_ORDER):
        patch.set_facecolor(SEASON_COLORS[season])
        patch.set_alpha(0.7)
    for i, season in enumerate(SEASON_ORDER):
        n = len(valid[valid['season'] == season])
        med = valid[valid['season'] == season]['hips_ftir_ratio'].median()
        ax.text(i + 1, ax.get_ylim()[1] * 0.95, f'n={n}\nmed={med:.2f}', ha='center', fontsize=8)
    ax.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='1:1')
    ax.set_ylabel('HIPS EC / FTIR EC', fontsize=11)
    ax.set_title('HIPS-FTIR Discrepancy by Season', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    # Panel 2: HIPS/FTIR ratio vs Precipitable Water
    ax = axes[1]
    valid_pw = filt_aero.dropna(subset=['hips_ftir_ratio', 'Precipitable_Water(cm)'])
    if len(valid_pw) > 3:
        for season in SEASON_ORDER:
            s = valid_pw[valid_pw['season'] == season]
            ax.scatter(s['Precipitable_Water(cm)'], s['hips_ftir_ratio'],
                      color=SEASON_COLORS[season], alpha=0.6, s=50, label=season,
                      edgecolors='k', linewidth=0.3)
        if len(valid_pw) > 5:
            r, p = stats.pearsonr(valid_pw['Precipitable_Water(cm)'], valid_pw['hips_ftir_ratio'])
            ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}', transform=ax.transAxes,
                    fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.axhline(y=1, color='k', linestyle='--', alpha=0.5)
    ax.set_xlabel('Precipitable Water (cm)', fontsize=11)
    ax.set_ylabel('HIPS EC / FTIR EC', fontsize=11)
    ax.set_title('HIPS-FTIR vs Column Moisture', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 3: HIPS/FTIR ratio vs Angstrom Exponent
    ax = axes[2]
    valid_ae = filt_aero.dropna(subset=['hips_ftir_ratio', '440-870_Angstrom_Exponent'])
    if len(valid_ae) > 3:
        for season in SEASON_ORDER:
            s = valid_ae[valid_ae['season'] == season]
            ax.scatter(s['440-870_Angstrom_Exponent'], s['hips_ftir_ratio'],
                      color=SEASON_COLORS[season], alpha=0.6, s=50, label=season,
                      edgecolors='k', linewidth=0.3)
        if len(valid_ae) > 5:
            r, p = stats.pearsonr(valid_ae['440-870_Angstrom_Exponent'], valid_ae['hips_ftir_ratio'])
            ax.text(0.05, 0.95, f'r={r:.3f}\np={p:.2e}', transform=ax.transAxes,
                    fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    ax.axhline(y=1, color='k', linestyle='--', alpha=0.5)
    ax.set_xlabel('Angstrom Exponent (440-870nm)', fontsize=11)
    ax.set_ylabel('HIPS EC / FTIR EC', fontsize=11)
    ax.set_title('HIPS-FTIR vs Particle Size\n(low AE = coarse-dominated)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.suptitle('HIPS-FTIR Discrepancy vs AERONET Column Properties (24h Filter Averages)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

---

## 6. AAE Data Quality Check

The diagnostics notebook showed surface AAE clustering near 0 and below. Negative AAE values suggest the UV/IR BC ratio is near 1 or inverted, possibly indicating a data quality issue with the C-correction. Let's investigate.

In [ ]:
# Investigate AAE calculation from sub-daily matched data
valid_ae = matched.dropna(subset=['UV_BCc', 'IR_BCc']).copy()
valid_ae = valid_ae[(valid_ae['UV_BCc'] > 0) & (valid_ae['IR_BCc'] > 0)]

# BC ratio (IR/UV) — the input to AAE
valid_ae['bc_ratio_IR_UV'] = valid_ae['IR_BCc'] / valid_ae['UV_BCc']

# AAE = log(IR/UV) / log(880/375)
wavelength_ratio = np.log(880 / 375)
valid_ae['AAE'] = np.log(valid_ae['bc_ratio_IR_UV']) / wavelength_ratio

print(f'BC ratio (IR/UV) statistics:')
print(f'  mean={valid_ae["bc_ratio_IR_UV"].mean():.3f}')
print(f'  median={valid_ae["bc_ratio_IR_UV"].median():.3f}')
print(f'  % > 1 (IR > UV, AAE < 0): {(valid_ae["bc_ratio_IR_UV"] > 1).mean()*100:.1f}%')
print(f'  % < 1 (UV > IR, AAE > 0): {(valid_ae["bc_ratio_IR_UV"] < 1).mean()*100:.1f}%')
print(f'\nAAE statistics:')
print(f'  mean={valid_ae["AAE"].mean():.3f}')
print(f'  median={valid_ae["AAE"].median():.3f}')
print(f'  % negative: {(valid_ae["AAE"] < 0).mean()*100:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: BC ratio histogram
ax = axes[0]
for season in SEASON_ORDER:
    s = valid_ae[valid_ae['season'] == season]['bc_ratio_IR_UV']
    ax.hist(s, bins=50, alpha=0.5, color=SEASON_COLORS[season], label=season, density=True)
ax.axvline(x=1, color='k', linestyle='--', linewidth=2, label='ratio=1 (AAE=0)')
ax.set_xlabel('IR BCc / UV BCc', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('BC Ratio Distribution\n(>1 means IR > UV → negative AAE)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: AAE distribution
ax = axes[1]
clipped = valid_ae['AAE'].clip(-2, 3)
for season in SEASON_ORDER:
    s = valid_ae[valid_ae['season'] == season]['AAE'].clip(-2, 3)
    ax.hist(s, bins=50, alpha=0.5, color=SEASON_COLORS[season], label=season, density=True)
ax.axvline(x=1, color='k', linestyle=':', alpha=0.7, label='AAE=1 (fossil fuel)')
ax.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='AAE=0')
ax.set_xlabel('Surface AAE (375-880nm)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('AAE Distribution\n(large negative tail)', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 3: Diurnal BC ratio
ax = axes[2]
for season in SEASON_ORDER:
    sdata = valid_ae[valid_ae['season'] == season]
    hourly = sdata.groupby('hour_local')['bc_ratio_IR_UV'].mean()
    ax.plot(hourly.index, hourly.values, 'o-', color=SEASON_COLORS[season],
            linewidth=2, markersize=6, label=season)
ax.axhline(y=1, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Hour (local)', fontsize=11)
ax.set_ylabel('IR BCc / UV BCc', fontsize=11)
ax.set_title('Diurnal BC Ratio\n(>1 = AAE negative, wavelength correction issue?)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('AAE Data Quality Investigation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 7. Regime Synthesis — FMF × BC × Filter Data

Classify days into four regimes (median split on BC and FMF) and connect each regime to:
- Filter measurements (HIPS, FTIR, Iron) averaged for that regime
- HIPS-FTIR discrepancy
- Season distribution

In [ ]:
# Build daily regime dataset: BC + AERONET SDA
bc_daily_df = pd.read_pickle(os.path.join(PROCESSED_SITES_DIR, 'df_Addis_Ababa_9am_resampled.pkl'))
bc_daily_df['datetime_local'] = pd.to_datetime(bc_daily_df['datetime_local'])
bc_daily_df.set_index('datetime_local', inplace=True)
bc_daily_df['IR BCc'] = bc_daily_df['IR BCc'] / 1000
bc_daily_df.loc[bc_daily_df['IR BCc'] < 0, 'IR BCc'] = np.nan

bc_for_merge = bc_daily_df[['IR BCc']].copy()
bc_for_merge.index = bc_for_merge.index.normalize()
if bc_for_merge.index.tz is not None:
    bc_for_merge.index = bc_for_merge.index.tz_localize(None)

regime_df = bc_for_merge.join(sda_daily[[coarse_col, fine_col, fmf_col]], how='inner')
regime_df = regime_df.join(aod_daily[['AOD_500nm', 'Precipitable_Water(cm)',
                                       '440-870_Angstrom_Exponent']], how='left')
regime_df = regime_df.dropna(subset=['IR BCc', fmf_col])
regime_df['month'] = regime_df.index.month
regime_df['season'] = regime_df['month'].map(assign_season)

bc_med = regime_df['IR BCc'].median()
fmf_med = regime_df[fmf_col].median()

def classify_regime(row):
    high_bc = row['IR BCc'] > bc_med
    high_fmf = row[fmf_col] > fmf_med
    if high_bc and high_fmf:
        return 'High BC + High FMF'
    elif high_bc and not high_fmf:
        return 'High BC + Low FMF'
    elif not high_bc and high_fmf:
        return 'Low BC + High FMF'
    else:
        return 'Low BC + Low FMF'

regime_df['regime'] = regime_df.apply(classify_regime, axis=1)

print(f'Regime analysis: {len(regime_df)} matched days')
print(f'  BC median: {bc_med:.3f} µg/m³,  FMF median: {fmf_med:.3f}')
print(f'\n{regime_df["regime"].value_counts()}')

In [ ]:
# Connect regime days to filter data
# For each filter date, assign the regime from that day
if filt_aero is not None:
    regime_lookup = regime_df[['regime']].copy()
    filt_aero['date_norm'] = pd.to_datetime(filt_aero['date']).dt.normalize()
    filt_aero = filt_aero.merge(
        regime_lookup, left_on='date_norm', right_index=True, how='left')

    n_with_regime = filt_aero['regime'].notna().sum()
    print(f'Filter samples with regime assignment: {n_with_regime} / {len(filt_aero)}')
    print(filt_aero['regime'].value_counts())

In [ ]:
# Regime synthesis plots
regime_order = ['High BC + High FMF', 'High BC + Low FMF',
                'Low BC + High FMF', 'Low BC + Low FMF']
regime_labels = ['High BC\nHigh FMF\n(combustion\nthroughout)',
                 'High BC\nLow FMF\n(BC trapped\nbelow dust)',
                 'Low BC\nHigh FMF\n(elevated\nfine layer)',
                 'Low BC\nLow FMF\n(clean/\ncoarse)']
regime_colors = ['#E74C3C', '#E67E22', '#3498DB', '#95A5A6']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel 1: BC vs FMF scatter with regime quadrants
ax = axes[0, 0]
for season in SEASON_ORDER:
    s = regime_df[regime_df['season'] == season]
    ax.scatter(s['IR BCc'], s[fmf_col], color=SEASON_COLORS[season],
              alpha=0.5, s=30, label=season, edgecolors='k', linewidth=0.3)
ax.axhline(y=fmf_med, color='gray', linestyle='--', alpha=0.7)
ax.axvline(x=bc_med, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('BC (µg/m³)', fontsize=10)
ax.set_ylabel('Fine Mode Fraction', fontsize=10)
ax.set_title('Regime Quadrants by Season', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 2: Regime distribution by season (stacked bar)
ax = axes[0, 1]
cross = pd.crosstab(regime_df['season'], regime_df['regime'], normalize='index') * 100
cross = cross.reindex(columns=regime_order)
cross.loc[SEASON_ORDER].plot(kind='bar', stacked=True, ax=ax, color=regime_colors)
ax.set_ylabel('% of days', fontsize=10)
ax.set_title('Regime Frequency by Season', fontsize=11, fontweight='bold')
ax.legend(fontsize=6, bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(SEASON_ORDER, rotation=0)
ax.grid(True, alpha=0.3, axis='y')

# Panel 3: Coarse AOD by regime
ax = axes[0, 2]
available_regimes = [r for r in regime_order if r in regime_df['regime'].values]
plot_data = [regime_df[regime_df['regime'] == r][coarse_col].dropna() for r in available_regimes]
if any(len(d) > 0 for d in plot_data):
    bp = ax.boxplot(plot_data, patch_artist=True, showfliers=False)
    ax.set_xticklabels([regime_labels[regime_order.index(r)] for r in available_regimes], fontsize=7)
    for i, r in enumerate(available_regimes):
        bp['boxes'][i].set_facecolor(regime_colors[regime_order.index(r)])
        bp['boxes'][i].set_alpha(0.7)
ax.set_ylabel('Coarse Mode AOD', fontsize=10)
ax.set_title('Coarse AOD by Regime', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Panel 4-6: Filter data by regime (if available)
if filt_aero is not None and 'regime' in filt_aero.columns:
    filt_with_regime = filt_aero.dropna(subset=['regime'])

    for i, (col, ylabel, title) in enumerate([
        ('hips_ftir_ratio', 'HIPS EC / FTIR EC', 'HIPS-FTIR Discrepancy'),
        ('ir_bcc', 'Aeth BC (µg/m³)', '24h Average BC'),
        ('iron', 'Iron (µg/m³)', 'Surface Iron (dust proxy)'),
    ]):
        ax = axes[1, i]
        if col in filt_with_regime.columns:
            available_r = [r for r in regime_order if r in filt_with_regime['regime'].values]
            plot_data = [filt_with_regime[filt_with_regime['regime'] == r][col].dropna()
                         for r in available_r]
            if any(len(d) > 0 for d in plot_data):
                bp = ax.boxplot(plot_data, patch_artist=True, showfliers=True,
                               flierprops=dict(marker='o', markersize=3, alpha=0.3))
                ax.set_xticklabels([regime_labels[regime_order.index(r)] for r in available_r], fontsize=7)
                for j, r in enumerate(available_r):
                    bp['boxes'][j].set_facecolor(regime_colors[regime_order.index(r)])
                    bp['boxes'][j].set_alpha(0.7)
                    n = len(filt_with_regime[(filt_with_regime['regime'] == r) & filt_with_regime[col].notna()])
                    ax.text(j + 1, ax.get_ylim()[1] * 0.95, f'n={n}', ha='center', fontsize=8)
            if col == 'hips_ftir_ratio':
                ax.axhline(y=1, color='k', linestyle='--', alpha=0.5)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(f'{title} by Regime', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Regime Synthesis: FMF × BC × Filter Data', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Regime summary table
print('=' * 90)
print('REGIME SUMMARY')
print('=' * 90)

summary_cols_regime = ['IR BCc', fmf_col, coarse_col, fine_col, 'AOD_500nm',
                        'Precipitable_Water(cm)', '440-870_Angstrom_Exponent']

for regime in regime_order:
    rdata = regime_df[regime_df['regime'] == regime]
    if len(rdata) == 0:
        continue
    print(f'\n{regime} (n={len(rdata)}):')
    season_dist = dict(rdata['season'].value_counts())
    print(f'  Seasons: {season_dist}')
    for col in summary_cols_regime:
        if col in rdata.columns:
            vals = rdata[col].dropna()
            if len(vals) > 0:
                print(f'  {col}: mean={vals.mean():.4f}, median={vals.median():.4f}')

    # Filter data for this regime
    if filt_aero is not None and 'regime' in filt_aero.columns:
        filt_r = filt_aero[filt_aero['regime'] == regime]
        for col in ['ir_bcc', 'hips_fabs', 'ftir_ec', 'iron', 'hips_ftir_ratio']:
            if col in filt_r.columns:
                vals = filt_r[col].dropna()
                if len(vals) > 0:
                    print(f'  [filter] {col}: mean={vals.mean():.3f}, n={len(vals)}')

---

## Summary & Narrative

### Key findings from this deep dive:

In [ ]:
print('=' * 80)
print('DEEP DIVE SUMMARY')
print('=' * 80)

print('''
1. DIURNAL DECOUPLING (headline):
   BC drops sharply 6am → midday; AOD stays flat or rises.
   This is textbook boundary layer trapping at 2370m.
   The surface BC morning spike is a trapping artifact, not an emissions spike.

2. BC/AOD RATIO:
   ~3-4x morning-to-afternoon swing = direct BL growth measurement.
   Kiremt consistently lower ratio = better atmospheric mixing.

3. FILTER DATA × AERONET:
   24h filter averages connected to daily columnar properties.
   Surface filter BC correlates with columnar AOD, but with high scatter
   due to the BL-driven decoupling.

4. DUST INTERFERENCE (null result):
   HIPS/FTIR ratio vs coarse AOD: r ≈ -0.33 (NEGATIVE).
   Higher dust → HIPS UNDERestimates vs FTIR.
   This ARGUES AGAINST dust causing the HIPS anomaly.

5. KIREMT COARSE MODE ANOMALY:
   Coarse AOD peaks in rainy season, not dry.
   Kiremt dominated by "High BC + Low FMF" regime.
   = surface BC trapped below a coarse-dominated column.

6. AAE DATA QUALITY:
   Many negative AAE values — IR BCc > UV BCc in a large fraction
   of observations. May indicate C-correction / wavelength correction
   issues in the aethalometer at this site.

7. OVERALL NARRATIVE:
   The Addis anomaly is primarily a boundary layer story.
   Strong diurnal BL dynamics at high elevation cause surface-column
   decoupling. HIPS (24h time-integrated surface absorption) captures
   a different signal than FTIR EC (mass-based). The BL trapping means
   surface measurements overweight morning conditions when absorption
   per unit mass is influenced by composition, mixing state, and coating.
''')

# Key correlations summary
print('--- Key correlations (24h filter × daily AERONET) ---')
if filt_aero is not None:
    test_pairs = [
        ('ir_bcc', 'AOD_500nm', 'BC vs AOD'),
        ('ir_bcc', fine_col, 'BC vs Fine AOD'),
        ('hips_ftir_ratio', coarse_col, 'HIPS/FTIR ratio vs Coarse AOD'),
        ('hips_ftir_ratio', fmf_col, 'HIPS/FTIR ratio vs FMF'),
        ('hips_ftir_ratio', 'Precipitable_Water(cm)', 'HIPS/FTIR ratio vs PW'),
        ('iron', coarse_col, 'Iron vs Coarse AOD'),
    ]
    for xcol, ycol, label in test_pairs:
        if xcol in filt_aero.columns and ycol in filt_aero.columns:
            valid = filt_aero.dropna(subset=[xcol, ycol])
            if len(valid) > 5:
                r, p = stats.pearsonr(valid[xcol], valid[ycol])
                sig = '*' if p < 0.05 else ''
                print(f'  {label}: r={r:.3f}{sig}, p={p:.2e}, n={len(valid)}')